<a href="https://colab.research.google.com/github/FinnLuetzen/KI-Vorlesung/blob/main/Aufgabe_1_Neuronale_Netzwerke.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aufgabe 1: Neuronale Netzwerke

Bibliotheken importieren (nicht bearbeiten):

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from google.colab import files

import warnings
warnings.filterwarnings('ignore')

Datensatz laden und vorbereiten (nicht bearbeiten):

In [ ]:
train_url = 'https://raw.githubusercontent.com/TNemes-3141/kiml-gn-public-datasets/refs/heads/main/task%201/diamonds_train.csv'
test_url = 'https://raw.githubusercontent.com/TNemes-3141/kiml-gn-public-datasets/refs/heads/main/task%201/diamonds_test.csv'

train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

print("Trainingsdatensatz (Ausschnitt):")
display(train_df.head())
print("Testdatensatz (Ausschnitt):")
display(test_df.head())

X_train_raw = train_df.drop(columns=['price'])
y_train = train_df['price'].values
X_test_raw = test_df.copy()

cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

encoder = OrdinalEncoder(categories=[cut_order, color_order, clarity_order])
cat_cols = ['cut', 'color', 'clarity']
num_cols = ['carat']

X_train_cat = encoder.fit_transform(X_train_raw[cat_cols])
X_test_cat = encoder.transform(X_test_raw[cat_cols])

X_train_processed = np.hstack((X_train_raw[num_cols].values, X_train_cat))
X_test_processed = np.hstack((X_test_raw[num_cols].values, X_test_cat))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_processed)
X_test_scaled = scaler.transform(X_test_processed)

Bauen Sie hier Ihr neuronales Netzwerk und legen Sie die Hyperparameter fest (HIER BEARBEITEN)!

❗ Sie müssen diesen Block bei jeder Änderung an Ihrem neuronalen Netz oder den Hyperparametern **neu ausführen**

In [ ]:
# === ZU BEARBEITENDER BLOCK ===

# --- 1. HYPERPARAMETER ---
LERNRATE = # ...
EPOCHEN = # ...
KOSTENFUNKTION = # ...

# --- 2. NEURONALES NETZWERK ---
# Fügen Sie hier die Schichten des Modells hinzu!
# Die Schichten müssen am Ende durch Kommas getrennt werden.
model = Sequential([
    # ...
    Dense(1, activation='linear'), # NICHT löschen - finaler Output-Layer!
])

# --- 3. MODELL KOMPILIEREN ---
# NICHT bearbeiten - das baut das Netzwerk für das Training zusammen.
model.compile(
    optimizer=Adam(learning_rate=LERNRATE),
    loss=KOSTENFUNKTION
)

print("Modell erfolgreich erstellt! Führen Sie den nächsten Block aus, um die Vorhersagen zu erstellen.")

Nun lassen wir das Modell anhand der Trainingsdaten lernen. Die generierte Grafik zeigt Ihnen, wie schnell Ihr Modell lernt. Sobald das Training abgeschlossen ist, wird das Modell die Preise für den Testsatz vorhersagen und diese zur Bewertung in einer Datei namens `result.csv` speichern.
(nicht bearbeiten)

In [ ]:
BATCH_SIZE = 128

print("Training wird gestartet...")
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=EPOCHEN,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=1
)

plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training-Loss (Fehler)')
plt.plot(history.history['val_loss'], label='Validation-Loss (Fehler)')
plt.title('Modell-Lernfortschritt')
plt.xlabel('Epoche')
plt.ylabel(f'Kosten ({KOSTENFUNKTION.upper()})')
plt.legend()
plt.grid(True)
plt.show()

print("\nTraining abgeschlossen! Vorhersagen für den Testsatz werden generiert...")
predictions = model.predict(X_test_scaled)

submission_df = pd.DataFrame({'price': predictions.flatten()})

submission_filename = 'result.csv'
submission_df.to_csv(submission_filename, index=False)

print("\nVorhersage abgeschlossen! Download...")
files.download(submission_filename)